In [0]:
%run ./01_setup_environment

In [0]:

# ========================================
# 05_silver_patients_transformation
# ========================================

from pyspark.sql.functions import *

try:

    patients_df = spark.read.format("delta") \
        .load(f"{bronze_path}/validated_patients")

    silver_patients_df = patients_df.dropDuplicates(["patient_id"])

    silver_patients_df = silver_patients_df.withColumn(
        "patient_name",
        initcap(trim(col("patient_name")))
    )

    silver_patients_df = silver_patients_df.withColumn(
        "city",
        initcap(trim(col("city")))
    )

    silver_patients_df = silver_patients_df.withColumn(
        "gender",
        upper(trim(col("gender")))
    )

    silver_patients_df = silver_patients_df.withColumn(
        "updated_at",
        current_timestamp()
    )

    silver_patients_df.write \
        .format("delta") \
        .mode("overwrite") \
        .save(f"{silver_path}/patients_clean")

    log_audit(
        "patients_pipeline",
        "silver",
        "patients_clean",
        silver_patients_df.count(),
        "SUCCESS"
    )

    print("Silver Patients Transformation Completed")

except Exception as e:

    log_audit(
        "patients_pipeline",
        "silver",
        "patients_clean",
        0,
        "FAILED",
        str(e)
    )

    raise e